# Video2Frames Prompt Tuning (APO) — reward v2

本 notebook 按 [README.md](README.md) 的流程，把每个运行步骤整理成可以逐个执行的 cell。
**本 notebook 固定使用 reward v2**（各命令显式传 `--reward-version v2`）；
reward v1 的对应流程见 `video2frames-prompt-tuning-reward-v1.ipynb`。

任务背景：用 Agent-Lightning 的 APO（自动提示词优化）算法，调优视频监控帧分析任务的固定指令 prompt。
客户数据集 `original_data/qwen_0318_swift_task.json` 含 5850 条视频分析任务（视频 → 结构化 JSON：`english_detail` / `brief` / `title` / `scene_type` / `is_courier_action`）。
每条视频任务被转换为**多帧图片任务**：去掉 `<video>` 占位符，在指令后追加逐帧占位符：

```
<frame 1 | 0s> <frame 2 | 3s> ... <frame n | 3(n-1)s>
```

APO 只调优 prompt 的**固定指令部分**；帧占位符部分在运行时按任务重建。

**Reward v2 与 v1 的差异**（详见文末附录与 `doc/reward-design.md` 第 5 节）：
分字段 LLM judge + 确定性规则合规组成软分，scene/courier 判错以**乘法硬门**作用于软分
（漏报真实快递员惩罚最重）。v1 仍是默认版本，作为可对比的基准刻度保留。

> **重要：** `original_data/qwen_0318_swift_task.json` 是客户数据，绝不能提交或推送到 GitHub。
> `original_data/`、`data/`、`log/`、`results/` 目录内容已被 git 忽略。
> 请单独（如 scp）把 `original_data/`、`data/` 和仓库根目录的 `.env` 拷到训练机器上。

## 0. 工作目录

确保 notebook 从仓库根目录（含 `frame_agent.py` 的目录）打开，后续所有命令都在该目录执行。

In [ ]:
import os

# notebook 应从仓库根目录打开；若不是，请先在 Jupyter 中切换目录
assert os.path.exists("frame_agent.py"), f"请从仓库根目录打开本 notebook（当前: {os.getcwd()}）"
print("当前工作目录:", os.getcwd())

# 强制用 .env 覆盖内核继承的环境变量。
# Jupyter server 启动时环境里可能残留旧的 AZURE_OPENAI_* / OPENAI_API_VERSION，
# 而代码内部的 load_dotenv(override=False) 不会覆盖它们，会导致 404 等诡异错误；
# 这里 override=True 保证后续 ! 子进程拿到的是 .env 里的值。
from dotenv import load_dotenv

load_dotenv(".env", override=True)
if not os.environ.get("OPENAI_API_VERSION") and os.environ.get("AZURE_OPENAI_API_VERSION"):
    os.environ["OPENAI_API_VERSION"] = os.environ["AZURE_OPENAI_API_VERSION"]
for key in ("AZURE_OPENAI_ENDPOINT", "OPENAI_API_VERSION", "AZURE_OPENAI_DEPLOYMENT"):
    print(key, "=", os.environ.get(key))

## 1. 安装

本项目必须运行在**从 [huqianghui/agent-lightning](https://github.com/huqianghui/agent-lightning) fork 源码构建的 agent-lightning 0.3.1** 上（PyPI 的 0.3.0 缺少所需功能）。`requirements.txt` 已按 commit 固定该 fork，直接安装即可。

In [ ]:
!python -m venv .venv
!.venv/bin/pip install -r requirements.txt

In [ ]:
# 期望输出 0.3.1
!.venv/bin/python -c "import agentlightning; print(agentlightning.__version__)"

## 2. 配置

Blob 存储配置从仓库根目录 `.env` 读取（`blob4videodatasets_connection_string`、
`blob4videodatasets_container_name`、`blob4videodatasets_frames_folder_name`）。
此外还需在 `.env` 中添加（或 export）以下 Azure OpenAI 变量：

| 变量 | 用途 | 默认值 |
| --- | --- | --- |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint URL | 必填 |
| `AZURE_OPENAI_API_KEY` | Azure OpenAI API key | 必填 |
| `OPENAI_API_VERSION` | Azure OpenAI API 版本（如 `2024-10-21`） | 必填 |
| `AZURE_OPENAI_DEPLOYMENT` | 分析帧所用的多模态 deployment | `gpt-4o` |
| `JUDGE_MODEL` | reward 中 LLM judge 使用的 deployment | `gpt-4.1-mini` |
| `APO_GRADIENT_MODEL` | APO 批评（critique）prompt 用的 deployment | `gpt-4.1` |
| `APO_APPLY_EDIT_MODEL` | APO 重写 prompt 用的 deployment | `gpt-4.1-mini` |
| `FRAMES_AS_BASE64` | 设为 `true` 时以 base64 data URI 发送帧（而非 SAS URL） | 未设置 |

In [ ]:
# 快速检查 .env 中的关键变量是否已配置（不打印值）
from pathlib import Path

env_path = Path(".env")
required = [
    "blob4videodatasets_connection_string",
    "blob4videodatasets_container_name",
    "blob4videodatasets_frames_folder_name",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "OPENAI_API_VERSION",
]
if env_path.exists():
    keys = {line.split("=", 1)[0].strip() for line in env_path.read_text().splitlines()
            if "=" in line and not line.strip().startswith("#")}
    for name in required:
        print(f"{'OK  ' if name in keys else 'MISS'} {name}")
else:
    print(f"未找到 {env_path.resolve()} — 请先准备 .env")

## 3. 准备数据集（Workflow 步骤 1）

> **与 v1 共用数据：** 数据准备与 reward 版本无关，`data/` 下的 split 两个版本共用。
> 若已在 v1 notebook 准备过数据、且希望 v1/v2 结果可直接对比（推荐），**跳过本节**复用同一份 split
> —— `apo_train.py` 会把 split 指纹（行数 + 哈希）记入 `summary.json`，便于事后核对两次运行用的是同一份数据。

分层抽样并从 Azure 解析帧 blob：

- 抽样与切分按 (family, `is_courier_action`) 联合分层，每个 split 都镜像候选池的标签分布；
  val split 保证至少 `--val-courier-min`（默认 0.15）的快递员正例，并记录各 split 的
  courier / scene_type / family 分布（scene_type 只报告分布，无配额）。
- `--probe-content-filter` 在抽样时用 Azure 内容安全过滤器探测每个候选视频
  （约 3% 的视频无论 prompt 如何都会被拒绝），被拦截的会回填替换，保证各 split 达到目标大小。
  探测结果按视频缓存在 `data/content_filter_cache.json`，重复运行不会重复探测。

In [ ]:
# 输出很长（azure blob 的每个请求都打 INFO 日志），重定向到文件，只看结尾摘要
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 40 --val-size 24 --test-size 30 --seed 42 --probe-content-filter \
    > log/prepare_data.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/prepare_data.log

### 3b.（第二轮）冻结 test、重新生成 train/val

`--freeze-test` 保持 `test.jsonl` 不动，并把其中的视频排除出重抽样（`--test-size` 被忽略）。
注意：即使 seed 相同，冻结 test 的运行也不会复现上一轮的 train/val（候选池已变化）。

In [ ]:
# 按需执行（第二轮扩大数据集时）；输出重定向到日志，只看结尾摘要
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 80 --val-size 100 --freeze-test --probe-content-filter \
    > log/prepare_data.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/prepare_data.log

## 4.（可选）审计已有 split 的内容安全过滤情况（Workflow 步骤 2）

In [ ]:
# 仅生成报告（data/content_filter_probe.json）；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

In [ ]:
# 报告 + 删除被拦截的任务（不回填）；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py --apply \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

In [ ]:
# 基于已有报告重新应用；输出重定向到日志
!mkdir -p log
!.venv/bin/python probe_content_filter.py --from-report \
    > log/probe_content_filter.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/probe_content_filter.log

## 5. 用基线 prompt 调试单条 rollout（Workflow 步骤 3）

In [ ]:
# 单条 rollout 调试；完整日志在 log/frame_agent_debug.log，页面显示尾部（含模型输出与 reward）
!mkdir -p log
!.venv/bin/python frame_agent.py --limit 1 --reward-version v2 \
    > log/frame_agent_debug.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -40 log/frame_agent_debug.log

## 6. 端到端冒烟测试 APO 循环（Workflow 步骤 4）

最小 beam（1/1/1），成本低，用于验证链路。注意：冒烟运行也会写自己的 `results/<run_id>/`，
并把 `results/latest` 指向它 —— 正式评估前务必确认 latest 指向的是全量运行。

In [ ]:
# 冒烟测试；apo_train 自己会写 log/apo_<run_id>.log，这里再把控制台输出也收进文件。
# 想实时看进度：在终端 tail -f log/apo_smoke_console.log
!mkdir -p log
!.venv/bin/python apo_train.py --smoke --reward-version v2 \
    > log/apo_smoke_console.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/apo_smoke_console.log

## 7. 全量 APO 运行（Workflow 步骤 5）

每次运行获得一个带时间戳的 run ID：

- APO 日志写入 `log/apo_<run_id>.log`，所有产物写入 `results/<run_id>/`；
- 产物包括：最优 prompt（`best_prompt.txt`）、完整优化报告（每轮候选 prompt、reward、
  gradient 批评、验证得分）`report.md` + `report.json`、精简版本树 `tree.md`、
  以及记录运行参数和 `data/` split 指纹（行数 + 哈希）的 `summary.json`；
- 当最优 prompt 优于种子时，`diffs.md` 展示其派生链每一步的 unified diff（如 v0 → v4 → v7）
  以及整体 seed → best 的 diff；
- 运行之间不会互相覆盖；`results/latest` 指向最新一次运行。

**Reward 版本固定：** `apo_train.py` 会把解析后的 reward 版本写入 `REWARD_VERSION` 环境变量，
保证 fork 出的 runner 进程用同一个 reward 打分，并把版本号与完整 reward 配置（权重、gate 比例、
judge prompt）记录到该次运行的 `summary.json` —— 事后可据此确认某次运行确实是 v2。

AgentOps SaaS 上传**默认关闭**（span 仍会本地追踪，APO 只需要这个）。
只有需要在 app.agentops.ai 回放会话时才加 `--enable-agentops-service`。

平台说明：Linux + Python ≤ 3.13 会自动并行（默认 `--n-runners 4`，可调）；
macOS / Windows 自动回退到串行 shm 模式（`n_runners=1`），命令相同、只是串行。
大规模运行请在 Linux 上执行；beam 超参与并发调优见 `doc/performance-tuning.md`。

In [ ]:
# 全量 APO（耗时数小时）；控制台输出重定向到文件，结束前页面不会有输出。
# 实时进度：在终端 tail -f log/apo_train_console.log（或 log/apo_<run_id>.log）
!mkdir -p log
!.venv/bin/python apo_train.py --reward-version v2 \
    > log/apo_train_console.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -30 log/apo_train_console.log

### 7b.（可选）从任意历史运行的日志重新生成报告

把下面的 `<run_id>` 替换成实际的 run ID。

In [ ]:
# 注意：apo_train.py 结束时会自动生成报告，此 cell 仅用于事后重新生成。
# 不指定 --log 时自动选最新的 log/apo_<run_id>.log；要指定某次运行，
# 把 RUN_ID 换成实际值（格式如 20260717_020449，可从 ls log/ 或 results/latest/summary.json 查）。
RUN_ID = "<run_id>"  # TODO: 替换为实际 run ID
!.venv/bin/python generate_report.py --log log/apo_{RUN_ID}.log --output-dir results/{RUN_ID} \
    > log/generate_report.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/generate_report.log

### 7c.（可选）仅从已有 report.md 构建版本树

不需要日志（例如从其他机器拷来的 `report.md`）。此模式下 beam 存活标记不可用。

In [ ]:
!.venv/bin/python generate_report.py --from-report results/latest/report.md \
    > log/generate_report.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/generate_report.log

## 8. 在保留 test split 上对比基线与调优 prompt（Workflow 步骤 6）

**运行 test 前逐项确认以下前置条件：**

1. **APO 确实产出了优于种子的 prompt** —— 检查 `results/<run_id>/report.md` 或日志结尾；
   若最优 prompt 仍是 v0（`Best prompt not updated`），跑 test 没有意义，先修数据/评估配置并重跑 APO。
2. **确认要评估的 `best_prompt.txt` 来自全量运行而非冒烟运行、且 reward 版本是 v2** ——
   检查该运行 `summary.json` 中的 beam 参数（冒烟是 1/1/1）与 `reward_version` 字段。
3. **test 必须保持未见** —— 优化期间绝不反复探测 test 分数；所有调优和 prompt 选择只用 val。
   每次基于 test 做的决策都会折损最终数字的可信度。在一轮优化收敛后只跑一次。

两次运行分别写出 `results/eval_baseline_v2.json` 和 `results/eval_tuned_v2.json`
（mean_reward + 每任务明细 + `component_stats`）。v2 的 `component_stats` 除各 judge
分量与 `rule_compliance` 外，还包含**各硬门的触发率**（`gate_scene_error` /
`gate_courier_false_positive` / `gate_courier_false_negative`）—— 只换 deployment 时
触发率下降，就是"瓶颈在感知而非 prompt"的直接证据（见 `doc/optimization-stages.md`）。

对比 `mean_reward` 即最终结论。若差距小于 val 上观察到的评估噪声，不要声称有提升 ——
先按 `doc/dataset-sizing.md` 扩大 split 再复核。

In [ ]:
!.venv/bin/python evaluate.py --name baseline_v2 --reward-version v2 \
    > log/eval_baseline_v2.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/eval_baseline_v2.log

In [ ]:
!.venv/bin/python evaluate.py --prompt results/latest/best_prompt.txt --name tuned_v2 --reward-version v2 \
    > log/eval_tuned_v2.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/eval_tuned_v2.log

In [ ]:
# 对比两次评估的 mean_reward（并展示各分量均值与硬门触发率）
import json
from pathlib import Path

for name in ("baseline_v2", "tuned_v2"):
    p = Path(f"results/eval_{name}.json")
    if not p.exists():
        print(f"{name:12s} 未找到 {p}")
        continue
    data = json.loads(p.read_text())
    print(f"{name:12s} mean_reward = {data.get('mean_reward')}")
    for comp, stats in (data.get("component_stats") or {}).items():
        print(f"    {comp:28s} mean={stats['mean']:.3f}  n={stats['n']}")

## 9.（可选）Reward v1 vs v2 对比（Workflow 步骤 7）

同一个 prompt 分别用 v1、v2 打分，再按任务逐条 join 两份结果 —— 用于校准两把"尺子"的刻度关系
（哪些任务 v2 因硬门大幅拉开、哪些任务两版一致）。完整的 2×2 端到端对比计划（v1-tuned / v2-tuned
× v1-score / v2-score）与已有实测记录见 `doc/reward-comparison.md`。

前一节已生成 `results/eval_baseline_v2.json`；这里补上 v1 刻度（若 v1 notebook 已跑过可跳过）。

In [ ]:
# 同一 baseline prompt 用 reward v1 打分（v1 notebook 已生成过则可跳过）
!.venv/bin/python evaluate.py --name baseline_v1 --reward-version v1 \
    > log/eval_baseline_v1.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/eval_baseline_v1.log

In [ ]:
# 按任务 join 两份结果：逐任务 delta 表、按 family 与整体的均值；
# join 后的 JSON 写在输入文件旁（results/compare_*.json）
!.venv/bin/python compare_rewards.py results/eval_baseline_v1.json results/eval_baseline_v2.json \
    > log/compare_rewards.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -40 log/compare_rewards.log

## 附：Reward 定义（v2）

reward 已抽取为版本化的 `reward/` 包（显式参数 > `REWARD_VERSION` 环境变量 > 默认 `v1`）；
v2 的权重、gate 比例与分字段 judge prompt 在 `reward/v2/config.yaml`，打分逻辑在 `reward/v2/reward.py`。

**软分（加权和）：**

- `0.45` × `judge_detail` —— `english_detail` 的事件覆盖 / 无幻觉 / 时序正确
- `0.20` × `judge_brief` —— 主体 + 动作准确，≤ 20 词
- `0.10` × `judge_title` —— 抓住最关键事件，≤ 6 词
- `0.25` × `rule_compliance` —— 一组确定性 0/1 检查的均值（零噪声），检查项直接来自
  baseline prompt 的硬性风格规则：恰好 5 个 JSON 键、字数上限（50/20/6）、
  用 "person" 而非性别词、不提 camera/frame/timestamp、Non-Notable 触发前缀一致

**乘法硬门（cap 而非权重，作用于软分）：**

- `scene_type` 判错 → 软分 × 0.5
- courier 误报（FP）→ 软分 × 0.3
- courier **漏报**（FN）→ 软分 × 0.2 —— 漏掉真实快递员是最贵的错误；
  FN < FP 的不对称假设需与客户确认

**硬性归零（与 v1 相同）：**

- 输出不是合法 JSON 对象 → 得 0 分
- 被 Azure OpenAI 内容安全过滤器拒绝 → 得 0 分（拒绝只取决于输入帧，对所有候选 prompt 相同）

**可选的 judge 降噪：** 在 `reward/v2/config.yaml` 设 `judge_samples: 3`（并把
`judge_temperature` 调为非零）即对每字段多次 judge 调用取中位数（约 √3 倍降噪、3 倍 judge 成本）。

为什么用乘法门而不是加权（结构 vs 权重、类别不平衡的统计可见性、感知边界下 gate 的三重角色）
见 `doc/reward-design.md` 第 5 节；v1 vs v2 实测刻度对比见 `doc/reward-comparison.md`。

## 附：APO 元提示词

APO 由两个元提示词驱动，在 `reward/v2/config.yaml` 的 `apo_meta_prompts` 段声明：

- *text gradient*（从 rollout trace 批评当前 prompt）描述优化目标即 reward 公式，
  **归 reward 版本所有** —— v2 用 `reward/v2/text_gradient_video2frames.poml`，
  向优化器说明分字段权重 + 硬门结构（尤其"漏报最贵"的阈值推动方向）；
- *apply edit*（重写 prompt）只编码与 reward 无关的不变量（5 字段 JSON 契约、
  禁止添加帧/`<video>` 占位符），共享默认文件 `prompts/apply_edit_video2frames.poml`。

传 `--default-poml` 可回退到框架内置模板。详见 `doc/apo-poml-customization.md`。

## 附：冒烟测试

### 离线（无网络、无凭据）

In [ ]:
!mkdir -p log
!.venv/bin/pytest tests/ -v > log/pytest.log 2>&1 && echo "== OK ==" || echo "== FAILED（看下方日志尾部）=="
!tail -20 log/pytest.log

### 在线（需要 blob 访问 + Azure OpenAI）

In [ ]:
!mkdir -p log
!.venv/bin/python prepare_data.py --train-size 2 --val-size 2 --test-size 2 \
    > log/smoke_online.log 2>&1 && echo "== prepare_data OK ==" || echo "== prepare_data FAILED =="
!.venv/bin/python frame_agent.py --limit 1 --reward-version v2 \
    >> log/smoke_online.log 2>&1 && echo "== frame_agent OK ==" || echo "== frame_agent FAILED =="
!.venv/bin/python apo_train.py --smoke --reward-version v2 \
    >> log/smoke_online.log 2>&1 && echo "== apo_train --smoke OK ==" || echo "== apo_train --smoke FAILED =="
!tail -30 log/smoke_online.log